# BERT Sentiment Analysis: SST-2

### Architecture
BERT (Bidirectional Encoder Representations from Transformers) with a sentiment classification head on top of [CLS] token output.

### Note: Training from Scratch

This notebook trains the BERT model from scratch on SST-2 English sentiment analysis data.

In practice, the proper approach would be to either:
1. Pre-train the model on a large corpus first (MLM + NSP), then fine-tune on this sentiment data
2. Load an existing pre-trained model (e.g. klue/bert-base, bert-base-uncased) and fine-tune on this data

However, for learning purposes, we train directly on the sentiment data without pre-training to understand the training pipeline and model structure.

### Task
Binary sentiment classification:
- Input : English sentence (e.g. "a masterpiece")
- Output: Positive (1) / Negative (0)

### Dataset
SST-2 (Stanford Sentiment Treebank)
- Train : 67,000 sentences
- Val   : 872 sentences
- Label : 0 (negative) / 1 (positive)

### Model Configuration
- d_model    : 768
- num_heads  : 12
- d_ff       : 3072
- num_layers : 12
- dropout    : 0.1
- max_seq_len: 128

### Key Design Decisions
- [CLS] token only used for classification
- segment_ids all zeros (single sentence, no NSP)
- MLM/NSP heads removed
- New classification head: Linear(768 → 2)
- Dropout added for regularization

## 1. Import

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('../../01_Transformer/transformer_pytorch/src')
sys.path.append('./src')

from bert import BERT
from tokenizer import Tokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Load Tokenizer

In [ ]:
en_tokenizer = Tokenizer()
en_tokenizer.load('../data/processed/en_tokenizer.model')
print(f"Vocab size: {en_tokenizer.vocab_size()}")

## 3. Load Dataset

In [ ]:
dataset = load_dataset("sst2")
print(dataset)
print(dataset['train'][0])

## 4. SST2Dataset

In [ ]:
class SST2Dataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len=128):
        """
        SST-2 dataset for BERT sentiment classification

        Args:
            data       : HuggingFace dataset split
            tokenizer  : trained SentencePiece tokenizer
            max_seq_len: maximum sequence length
        """
        self.data        = data
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.data[idx]['sentence']
        label    = self.data[idx]['label']

        # Tokenize: [CLS] sentence [SEP]
        token_ids   = self.tokenizer.encode(sentence)[:self.max_seq_len]
        segment_ids = [0] * len(token_ids)  # all zeros — single sentence

        return (torch.tensor(token_ids),
                torch.tensor(segment_ids),
                torch.tensor(label))


def collate_fn(batch, pad_idx=0):
    """
    Pad sequences to max length in batch and generate padding mask
    """
    token_batch, segment_batch, label_batch = zip(*batch)

    max_len        = max(t.size(0) for t in token_batch)
    tokens_padded  = torch.full((len(token_batch), max_len), pad_idx, dtype=torch.long)
    segment_padded = torch.full((len(token_batch), max_len), 0,       dtype=torch.long)

    for i, (tokens, segment) in enumerate(zip(token_batch, segment_batch)):
        tokens_padded [i, :tokens.size(0)]  = tokens
        segment_padded[i, :segment.size(0)] = segment

    # Padding mask: (batch, 1, 1, seq_len)
    mask   = (tokens_padded != pad_idx).unsqueeze(1).unsqueeze(2)
    labels = torch.tensor(label_batch, dtype=torch.long)

    return tokens_padded, segment_padded, mask, labels


train_loader = DataLoader(
    SST2Dataset(dataset['train'], en_tokenizer, max_seq_len=128),
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, en_tokenizer.PAD_IDX)
)

val_loader = DataLoader(
    SST2Dataset(dataset['validation'], en_tokenizer, max_seq_len=128),
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, en_tokenizer.PAD_IDX)
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")

## 5. Sentiment Classifier

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, bert, d_model=768, num_classes=2, dropout=0.1):
        """
        BERT + Classification head

        Args:
            bert       : BERT model
            d_model    : BERT hidden dimension (768)
            num_classes: number of classes (2: positive/negative)
            dropout    : dropout rate
        """
        super().__init__()
        self.bert       = bert
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, segment_ids, mask=None):
        # BERT forward — only use encoder output (ignore MLM/NSP heads)
        x = self.bert.embedding(x, segment_ids)
        x = self.bert.encoder(x, mask)

        # Use [CLS] token (position 0) for classification
        cls_output = x[:, 0, :]                  # (batch, d_model)
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)  # (batch, num_classes)

        return logits


bert  = BERT(vocab_size=en_tokenizer.vocab_size()).to(device)
model = SentimentClassifier(bert).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 6.  Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-5,      # smaller lr for fine-tuning style training
    betas=(0.9, 0.999),
    eps=1e-8
)

## 7. Training & Validation Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch_idx, (tokens, segment_ids, mask, labels) in enumerate(loader):
        tokens      = tokens.to(device)
        segment_ids = segment_ids.to(device)
        mask        = mask.to(device)
        labels      = labels.to(device)

        logits = model(tokens, segment_ids, mask)
        loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss    += loss.item()
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total_samples += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Acc: {total_correct/total_samples:.4f}")

    return total_loss / len(loader), total_correct / total_samples


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0

    with torch.no_grad():
        for tokens, segment_ids, mask, labels in loader:
            tokens      = tokens.to(device)
            segment_ids = segment_ids.to(device)
            mask        = mask.to(device)
            labels      = labels.to(device)

            logits = model(tokens, segment_ids, mask)
            loss   = criterion(logits, labels)

            total_loss    += loss.item()
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total_samples += labels.size(0)

    return total_loss / len(loader), total_correct / total_samples

## 8. Run Training

In [ ]:
os.makedirs('../data/checkpoints', exist_ok=True)

num_epochs     = 10
best_val_acc   = 0.0
train_losses, val_losses     = [], []
train_accs,   val_accs       = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss, train_acc = train_epoch(model, train_loader,
                                        optimizer, criterion, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f} | Val Acc  : {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'             : val_acc,
            'train_losses'        : train_losses,
            'val_losses'          : val_losses,
            'train_accs'          : train_accs,
            'val_accs'            : val_accs,
        }, '../data/checkpoints/best_sentiment_model.pt')
        print(f"  ✅ Best model saved (val_acc: {val_acc:.4f})")

## 9. Loss & Accuracy Curve

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses,   label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs,   label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curve')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 10. Inference

In [ ]:
def predict(model, sentence, tokenizer, device):
    model.eval()

    token_ids   = tokenizer.encode(sentence)[:128]
    segment_ids = [0] * len(token_ids)

    tokens      = torch.tensor(token_ids).unsqueeze(0).to(device)
    segment_ids = torch.tensor(segment_ids).unsqueeze(0).to(device)
    mask        = (tokens != tokenizer.PAD_IDX).unsqueeze(1).unsqueeze(2)

    with torch.no_grad():
        logits = model(tokens, segment_ids, mask)
        pred   = logits.argmax(dim=-1).item()

    return "Positive 😊" if pred == 1 else "Negative 😞"


# Test
sentences = [
    "This movie was absolutely amazing!",
    "What a waste of time and money.",
    "The acting was brilliant and the story was compelling.",
    "I fell asleep halfway through the film.",
]

for sentence in sentences:
    result = predict(model, sentence, en_tokenizer, device)
    print(f"Input : {sentence}")
    print(f"Output: {result}")
    print()